# Machine Learning Zoomcamp

## 2. Machine Learning for Regression — Practice

Work through each exercise from memory before running it. If you get stuck, peek at the
corresponding section in the solution notebook: `02-regression.ipynb` (same folder).

Dataset: [Car price data](https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv)

Plan:

* Data preparation
* Exploratory data analysis
* Setting up the validation framework
* Linear regression (simple form)
* Linear regression (vector form)
* Training linear regression: normal equation
* Baseline model for car price prediction
* RMSE
* Validating the model
* Feature engineering
* Categorical variables
* Regularization
* Tuning the model
* Using the model


In [ ]:
# Import pandas and numpy under their usual aliases


## 2.2 Data preparation

1. Download the car price dataset from the URL above and read it into a DataFrame called `df`.
2. Normalize the column names: lowercase them and replace spaces with underscores.
3. Find the columns with string (`object`) dtype.
4. Normalize the *values* in those string columns too: lowercase and replace spaces with underscores.
5. Check `df.dtypes` to confirm the cleanup worked.


In [ ]:
# 1-2. Download and read the dataset, normalize column names


In [ ]:
# 3. Find columns with object dtype


In [ ]:
# 4. Normalize string values in those columns


In [ ]:
# 5. Check df.dtypes


## 2.3 Exploratory data analysis

1. For each column, print its name, the first 5 unique values, and the number of unique values.
2. Plot the distribution of `msrp` (the price column) with a histogram (50 bins).
3. Plot the same distribution but only for `msrp < 100000` -- notice the long tail.
4. Apply `np.log1p()` to `msrp` and plot the histogram of the transformed values. Compare the shape
   to the untransformed distribution -- why do we prefer this shape for a regression target?
5. Check `df.isnull().sum()` to see which columns have missing values.

**Recall:** why do long-tail target distributions confuse ML models, and why does `log1p` (rather
than plain `log`) matter here?


In [ ]:
# 1. Per-column unique value summary


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


In [ ]:
# 2. Histogram of msrp


In [ ]:
# 3. Histogram of msrp < 100000


In [ ]:
# 4. log1p transform + histogram


In [ ]:
# 5. Missing value counts


## 2.4 Setting up the validation framework

1. Compute `n_val`, `n_test`, `n_train` as a 20% / 20% / 60% split of the dataset (in that order --
   validation, test, train).
2. Build the row indices with `np.arange(n)`, seed the random generator (`np.random.seed(2)`) and
   shuffle the indices with `np.random.shuffle()`.
3. Use the shuffled indices to slice out `df_train`, `df_val`, `df_test`.
4. Reset the index on each of the three DataFrames (`reset_index(drop=True)`).
5. Build `y_train`, `y_val`, `y_test` as the log1p-transformed `msrp` column from each split.
6. Delete the `msrp` column from `df_train`, `df_val`, `df_test` -- why do we have to do this before
   training?

**Recall:** why do we shuffle before splitting, and why must we fix the seed?


In [ ]:
# 1. Compute split sizes


In [ ]:
# 2. Shuffle row indices with a fixed seed


In [ ]:
# 3. Slice df_train / df_val / df_test using the shuffled indices


In [ ]:
# 4. Reset indices


In [ ]:
# 5. Build y_train / y_val / y_test (log1p of msrp)


In [ ]:
# 6. Delete msrp from each split


## 2.5 Linear regression (simple form)

1. Pick one row from `df_train` (e.g. `df_train.iloc[10]`) and note its feature values.
2. Write a `linear_regression(xi)` function that takes a list of feature values `xi` and returns
   `w0 + sum(w[j] * xi[j] for j in range(len(xi)))`, using some made-up `w0` and `w` (e.g. `w0 = 7.17`,
   `w = [0.01, 0.04, 0.002]`).
3. Call it on a sample `xi` and note the prediction is in log scale.
4. Use `np.expm1()` to convert the log-scale prediction back to the original price scale.

**Recall:** what does `w0` represent when all the features are zero?


In [ ]:
# 1. Inspect a training row


In [ ]:
# 2. Define linear_regression(xi) -- the loop version


In [ ]:
# 3-4. Predict on a sample xi, then undo the log1p transform


## 2.6 Linear regression (vector form)

1. Write a `dot(xi, w)` helper that computes the dot product of two equal-length lists by hand
   (a loop, no numpy).
2. Rewrite `linear_regression(xi)` in terms of `dot(xi, w)`.
3. Fold the bias into the weight vector: build `w_new = [w0] + w`, and rewrite `linear_regression(xi)`
   so it prepends a `1` to `xi` and calls `dot(xi, w_new)` -- no separate `w0` term needed.
4. Stack several feature rows into a matrix `X` (`np.array` of rows, each starting with a `1` for the
   bias) and rewrite `linear_regression(X)` as a single `X.dot(w_new)` call.

**Recall:** why does prepending a constant `1` to every row let us drop the separate `w0` term?


In [ ]:
# 1. dot(xi, w)


In [ ]:
# 2-3. linear_regression via dot(), then via the folded-in bias trick


In [ ]:
# 4. Stack rows into X and predict for all of them with one matrix multiply


## 2.7 Training a linear regression model (normal equation)

1. Build a small feature matrix `X` (a handful of rows, a few numeric columns) and a target vector `y`.
2. Add a column of ones to `X` (the bias column) using `np.column_stack`.
3. Compute the Gram matrix `XTX = X.T.dot(X)`, invert it with `np.linalg.inv`, and compute
   `w_full = XTX_inv.dot(X.T).dot(y)` -- this is the normal equation w = (X^T X)^-1 X^T y.
4. Split `w_full` into `w0 = w_full[0]` and `w = w_full[1:]`.
5. Wrap all of the above into a `train_linear_regression(X, y)` function that returns `(w0, w)`.

**Recall:** why can't we just invert `X` directly instead of going through `X.T.dot(X)`?


In [ ]:
# 1-4. Build X, y, prepend bias column, solve the normal equation by hand


In [ ]:
# 5. Wrap it into train_linear_regression(X, y)


## 2.8 Baseline model for car price prediction

1. Pick a `base` list of 5 numeric columns from `df_train` (e.g. `engine_hp`, `engine_cylinders`,
   `highway_mpg`, `city_mpg`, `popularity`).
2. Build `X_train` from those columns, filling missing values with `0` for now.
3. Train the model with `train_linear_regression(X_train, y_train)` to get `w0, w`.
4. Compute predictions `y_pred = w0 + X_train.dot(w)`.
5. Plot `y_pred` and `y_train` on the same histogram (different colors, `alpha=0.5`) to eyeball the fit.

**Recall:** why is filling missing values with `0` a simplification rather than a good default --
what would be a better filler value, and why does that also need care (train vs. val leakage)?


In [ ]:
# 1-3. Build base feature list, X_train, train the model


In [ ]:
# 4-5. Predict and compare y_pred vs y_train visually


## 2.9 RMSE

1. Write an `rmse(y, y_pred)` function implementing the root mean squared error:
   sqrt(mean((g(x_i) - y_i)^2)).
2. Compute the RMSE of the baseline model's predictions on the training set.

**Recall:** what does a lower RMSE mean, and why do we care about evaluating on data the model
hasn't been trained on?


In [ ]:
# 1. def rmse(y, y_pred)


In [ ]:
# 2. RMSE on the training predictions


## 2.10 Validating the model

1. Write a `prepare_X(df)` function that selects the `base` columns, fills missing values with `0`,
   and returns a numpy array -- this avoids repeating the same feature-prep logic everywhere.
2. Use `prepare_X` + `train_linear_regression` to train on `df_train`.
3. Use `prepare_X` on `df_val`, predict, and compute the validation RMSE.

**Recall:** why does wrapping feature prep in a function matter once you also need to apply it to
`df_val` and `df_test`, not just `df_train`?


In [ ]:
# 1. prepare_X(df)


In [ ]:
# 2-3. Train on df_train, evaluate RMSE on df_val


## 2.11 Simple feature engineering

1. Extend `prepare_X` to add an `age` feature: `2017 - year`.
2. Retrain and re-evaluate the validation RMSE -- did it improve?
3. Plot predicted vs. actual `y_val` on the same histogram with a legend.

**Recall:** why might car age be predictive of price beyond what's already captured by the other
numeric features?


In [ ]:
# 1-2. Add age to prepare_X, retrain, re-evaluate RMSE


In [ ]:
# 3. Plot y_pred vs y_val with a legend


## 2.12 Categorical variables

1. Pick a handful of categorical columns (e.g. `make`, `model`, `engine_fuel_type`, `driven_wheels`,
   `market_category`, `vehicle_size`, `vehicle_style`).
2. For each one, find its top-5 most frequent values in `df_train` (`.value_counts().head()`).
3. Extend `prepare_X` to one-hot encode: for each categorical column and each of its top values,
   add a `0`/`1` column indicating whether that row has that value. Also one-hot encode
   `number_of_doors` for values `2`, `3`, `4`.
4. Retrain and check the validation RMSE -- did adding categoricals help or hurt?

**Recall:** why do we only keep the *top-5* values per category instead of one-hot encoding every
distinct value?


In [ ]:
# 1-2. Pick categorical columns, find top-5 values per column in df_train


In [ ]:
# 3. Extend prepare_X with one-hot encoded categoricals + num_doors


In [ ]:
# 4. Retrain and re-check validation RMSE


## 2.13 Regularization

1. Construct a small `X` where two columns are (near-)duplicates (one differs by `0.00000001`).
   Compute `XTX = X.T.dot(X)` and try to invert it -- what happens numerically?
2. Add a small value to the diagonal of `XTX` (`XTX + r * np.eye(...)`) before inverting -- does the
   inversion become well-behaved?
3. Write `train_linear_regression_reg(X, y, r=0.001)` -- the same normal equation as before, but with
   the diagonal regularization term added to `XTX` before inverting.
4. Retrain the full-feature model with `train_linear_regression_reg` at some fixed `r` and check the
   validation RMSE.

**Recall:** in your own words, why does adding a small constant to the diagonal fix the
near-singular matrix problem? What is this technique called?


In [ ]:
# 1. Build a near-duplicate-column X, inspect XTX and try to invert it


In [ ]:
# 2. Add r * np.eye(...) to the diagonal and invert again


In [ ]:
# 3. def train_linear_regression_reg(X, y, r=0.001)


In [ ]:
# 4. Retrain with regularization and check validation RMSE


## 2.14 Tuning the model

1. Loop over a range of `r` values (e.g. `[0.0, 0.00001, 0.0001, 0.001, 0.1, 1, 10]`), retrain with
   each, and print `r`, `w0`, and the validation RMSE for each.
2. Pick the `r` with the best (lowest) validation RMSE and note it down.

**Recall:** why might `r=0` sometimes look fine here even though we showed a case earlier where an
un-regularized normal equation blew up?


In [ ]:
# 1. Loop over candidate r values, print r / w0 / validation RMSE for each


In [ ]:
# 2. Note down the best r


## 2.15 Using the model

1. Combine `df_train` and `df_val` into `df_full_train` (`pd.concat`, then reset the index) and
   likewise concatenate `y_train` and `y_val` into `y_full_train`.
2. Train the final model on the combined data with your chosen `r`.
3. Evaluate the final RMSE on the untouched `df_test` / `y_test`.
4. Pick a single row from `df_test`, wrap it in a one-row DataFrame, run it through `prepare_X`, and
   predict its price. Undo the log transform with `np.expm1()` and compare to the actual test price
   (also un-logged).

**Recall:** why do we only touch `df_test` once, right at the very end?


In [ ]:
# 1. Build df_full_train / y_full_train


In [ ]:
# 2-3. Train the final model, evaluate RMSE on df_test


In [ ]:
# 4. Predict a single test row and compare to its actual (un-logged) price


## Wrap-up

In your own words (no code), answer:

1. What problem does the validation framework solve that a single train/test split doesn't?
2. Walk through the normal equation from memory: what are the shapes of `X`, `XTX`, and `w`?
3. What's the difference between feature engineering (age, one-hot categories) and regularization --
   what problem does each one solve?
